# Stratified Random Sampling Workflow for Classification Validation

This notebook implements a complete workflow for stratified random sampling following Olofsson et al. (2014) recommendations:

1. **Count pixels per class** from input rasters
2. **Design sampling strategy** by determining sample size and allocation per class
3. **Execute stratified sampling** to generate validation points in EPSG:4326 for platforms like Collect Earth Online

The workflow ensures proper statistical rigor with configurable parameters for user accuracies, standard errors, and sample allocation strategies.

## Import Libraries

In [1]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import rioxarray as rioxr
from random import sample
from shapely.geometry import Point
from rasterio.crs import CRS

## Configuration Parameters

Configure all parameters for the workflow in this section.

In [2]:
# ============================================================
# STEP 1: PIXEL COUNTING CONFIGURATION
# ============================================================

# Input rasters for pixel counting and sampling
# Year of the classification map to sample
YEAR = 2020

RASTER_FILES = [f"{YEAR}_rawmodel.tif"]

# Class definitions in rasters:
# 1 = iceplant
# 2 = other (non-vegetation)
# 3 = other vegetation
CLASS_MAPPING = {
    1: 'n_ice',      # iceplant
    2: 'n_other',    # other/non-vegetation
    3: 'n_ovg'       # other vegetation
}

# ============================================================
# STEP 2: SAMPLING DESIGN CONFIGURATION
# ============================================================

# Standard error for overall accuracy that we would like to achieve
# Example: 0.0105 means 95% confidence interval of ±2.06% around overall accuracy
STD_ERROR = 0.0105

# Estimated user's accuracies (precision) per class: TP/(TP+FP)
# IMPORTANT: Order must match CLASS_MAPPING order (n_ice, n_other, n_ovg)
# [iceplant, other/non-vegetation, other vegetation]
USER_ACCURACIES = {
    'n_ice': 0.88,    # iceplant
    'n_other': 0.94,     # other/non-vegetation
    'n_ovg': 0.91     # other vegetation
}

# Fixed allocation configuration
# Classes that should receive a fixed number of samples (e.g., rare classes)
FIXED_ALLOCATION_CLASSES = ['n_ovg', 'n_ice']  # vegetation and iceplant

# Different fixed sample sizes to explore for fixed allocation classes
FIXED_SAMPLES_OPTIONS = [300, 200, 150, 100]

# ============================================================
# STEP 3: STRATIFIED SAMPLING EXECUTION
# ============================================================

# Output configuration
# The sample is drawn without a fixed random seed, so a re-run produces a new set of points.
# The reference points used in the paper are provided in ../data/.
SAVE_POINTS = False
OUTPUT_FILENAME = f'validation_sample_points_{YEAR}.csv'

---
## Step 1: Count Pixels Per Class

Count the number of pixels in each class across all input rasters.

In [3]:
def count_pixels_per_class(raster_files, class_mapping):
    """
    Count pixels per class from a list of raster files.
    
    Parameters
    ----------
    raster_files : list
        List of raster file paths to process
    class_mapping : dict
        Dictionary mapping class values to class names
        
    Returns
    -------
    pd.DataFrame
        DataFrame with pixel counts per class for each raster
    pd.DataFrame
        DataFrame with total pixel counts across all rasters
    """
    print("Counting pixels per class...")
    
    # Initialize storage for pixel counts
    pixel_counts = {class_name: [] for class_name in class_mapping.values()}
    raster_names = []
    
    # Process each raster
    for raster_file in raster_files:
        print(f"  Processing: {raster_file}")
        raster = rioxr.open_rasterio(raster_file).squeeze()
        
        # Count pixels for each class
        for class_val, class_name in class_mapping.items():
            count = len(np.where(raster == class_val)[0])
            pixel_counts[class_name].append(count)
        
        # Store raster identifier
        raster_name = os.path.splitext(os.path.basename(raster_file))[0]
        raster_names.append(raster_name)
    
    # Create DataFrame with counts per raster
    df_per_raster = pd.DataFrame(pixel_counts)
    df_per_raster['raster'] = raster_names
    
    # Create DataFrame with total counts
    df_total = pd.DataFrame([df_per_raster[list(class_mapping.values())].sum()])
    
    print(f"\nTotal pixels counted: {df_total.sum(axis=1).values[0]:,}\n")
    
    return df_per_raster, df_total

In [4]:
# Execute pixel counting
df_per_raster, df_total = count_pixels_per_class(RASTER_FILES, CLASS_MAPPING)

print("Pixel counts per raster:")
display(df_per_raster)

print("\nTotal pixel counts:")
display(df_total)

Counting pixels per class...
  Processing: 2020_rawmodel.tif

Total pixels counted: 29,707,461

Pixel counts per raster:


,n_ice,n_other,n_ovg,raster
0,6861874,8574880,14270707,2020_rawmodel



Total pixel counts:


,n_ice,n_other,n_ovg
0,6861874,8574880,14270707


---
## Step 2: Sampling Design

### 2.1 Determine Sample Size

Following Olofsson et al. (2014, Eq. 13), based on Cochran (1977, Eq. 5.25):

$$n \approx \left( \frac{\sum W_iS_i}{S(\hat{O})}\right)^2$$

Where:
- $S(\hat{O})$ is the standard error of the estimated overall accuracy we want to achieve
- $W_i$ is the fraction of the map labeled as class $i$
- $S_i = \sqrt{U_i(1-U_i)}$ is the standard deviation of class $i$
- $U_i$ is the user's accuracy for class $i$ (precision: TP/(TP+FP))

In [5]:
def calculate_sample_size(df_total, user_accuracies_dict, std_error):
    """
    Calculate required sample size using Cochran's formula.
    
    Parameters
    ----------
    df_total : pd.DataFrame
        DataFrame with total pixel counts per class
    user_accuracies_dict : dict
        Dictionary mapping class names to estimated user's accuracies
    std_error : float
        Desired standard error of estimated overall accuracy
        
    Returns
    -------
    int
        Required sample size (rounded up)
    list
        Pixel proportions per class (order matches df_total columns)
    """
    # Calculate total pixels and proportions
    total_pixels = df_total.sum(axis=1).iloc[0]
    pixel_proportions = [df_total[col].iloc[0] / total_pixels for col in df_total.columns]
    
    # Get user accuracies in the same order as df_total columns
    user_accuracies = [user_accuracies_dict[col] for col in df_total.columns]
    
    # Calculate standard deviations per class (Cochran 1977, Eq. 5.55)
    std_devs = [np.sqrt(u * (1 - u)) for u in user_accuracies]
    
    # Calculate sample size
    numerator = sum([w * s for w, s in zip(pixel_proportions, std_devs)])
    sample_size = int((numerator / std_error) ** 2)
    
    # Print results
    print(f"Sample Size Calculation:")
    print(f"  Standard Error: {std_error}")
    print(f"  95% Confidence Interval: ±{std_error * 1.96 * 100:.2f}%")
    print(f"  Required Sample Size: {sample_size}\n")
    
    return sample_size, pixel_proportions

In [6]:
# Calculate sample size
sample_size, pixel_proportions = calculate_sample_size(
    df_total, 
    USER_ACCURACIES, 
    STD_ERROR
)

Sample Size Calculation:
  Standard Error: 0.0105
  95% Confidence Interval: ±2.06%
  Required Sample Size: 716



### 2.2 Determine Sample Allocation Per Class

Sample allocation strategies range from:
- **Proportional**: Allocate samples proportionally to class area (may undersample rare classes)
- **Equal**: Divide samples equally among classes (not optimal for area/overall accuracy)
- **Fixed + Proportional**: Allocate fixed samples to rare classes, then distribute remaining proportionally

We'll explore multiple allocation strategies and compute their confidence intervals.

In [7]:
def sample_allocation(df_total, fixed_classes, fixed_n, sample_size):
    """
    Allocate samples per class using fixed + proportional strategy.
    
    Parameters
    ----------
    df_total : pd.DataFrame
        DataFrame with total pixel counts per class
    fixed_classes : list
        List of class names that receive fixed allocation
    fixed_n : int
        Number of samples for each fixed class
    sample_size : int
        Total number of samples to allocate
        
    Returns
    -------
    list
        Number of samples per class (order matches df_total columns)
    """
    # Classes with proportional allocation
    prop_classes = [col for col in df_total.columns if col not in fixed_classes]
    
    # Calculate proportions for non-fixed classes
    total_prop_pixels = df_total[prop_classes].sum(axis=1).iloc[0]
    proportions = {col: df_total[col].iloc[0] / total_prop_pixels for col in prop_classes}
    
    # Remaining samples after fixed allocation
    remaining_samples = sample_size - (fixed_n * len(fixed_classes))
    
    # Allocate samples
    allocation = []
    for col in df_total.columns:
        if col in fixed_classes:
            allocation.append(fixed_n)
        else:
            allocation.append(int(proportions[col] * remaining_samples))
    
    return allocation


def calculate_confidence_intervals(user_accuracies_dict, sample_allocation, class_names):
    """
    Calculate 95% confidence interval radius for user's accuracies.
    
    Parameters
    ----------
    user_accuracies_dict : dict
        Dictionary mapping class names to estimated user's accuracies
    sample_allocation : list
        Number of samples per class
    class_names : list
        List of class names (order matches sample_allocation)
        
    Returns
    -------
    list
        95% CI radius (as percentage) for each class
    """
    # Get user accuracies in the same order as sample_allocation
    user_accuracies = [user_accuracies_dict[name] for name in class_names]
    
    # Calculate standard errors (Olofsson et al. 2014, Eq. 6)
    std_errors = [np.sqrt(u * (1 - u) / (n - 1)) for u, n in zip(user_accuracies, sample_allocation)]
    
    # 95% confidence interval radius (±1.96 * SE)
    ci_radius = [np.round(1.96 * se * 100, 2) for se in std_errors]
    
    return ci_radius

In [8]:
# Explore different allocation strategies
print("Exploring Sample Allocation Strategies...\n")

allocation_strategies = {}
confidence_intervals = {}

# Strategy 1: Equal allocation
n_classes = len(df_total.columns)
equal_alloc = [sample_size // n_classes] * n_classes
allocation_strategies['equal'] = equal_alloc
confidence_intervals['equal'] = calculate_confidence_intervals(USER_ACCURACIES, equal_alloc, df_total.columns)

# Strategy 2: Fixed allocation for specified classes with various fixed values
for fixed_n in FIXED_SAMPLES_OPTIONS:
    strategy_name = f'fixed_{fixed_n}'
    alloc = sample_allocation(df_total, FIXED_ALLOCATION_CLASSES, fixed_n, sample_size)
    allocation_strategies[strategy_name] = alloc
    confidence_intervals[strategy_name] = calculate_confidence_intervals(USER_ACCURACIES, alloc, df_total.columns)

# Strategy 3: Proportional allocation
prop_alloc = [int(sample_size * p) for p in pixel_proportions]
allocation_strategies['proportional'] = prop_alloc
confidence_intervals['proportional'] = calculate_confidence_intervals(USER_ACCURACIES, prop_alloc, df_total.columns)

# Create comparison DataFrames
df_allocation = pd.DataFrame(allocation_strategies, index=df_total.columns)
df_ci = pd.DataFrame(confidence_intervals, index=df_total.columns)

print("Sample Allocation Per Class:")
display(df_allocation)

print("\n95% Confidence Interval Radius (%) for User's Accuracies:")
display(df_ci)

Exploring Sample Allocation Strategies...

Sample Allocation Per Class:


,equal,fixed_300,fixed_200,fixed_150,fixed_100,proportional
n_ice,238,300,200,150,100,165
n_other,238,116,316,416,516,206
n_ovg,238,300,200,150,100,343



95% Confidence Interval Radius (%) for User's Accuracies:


,equal,fixed_300,fixed_200,fixed_150,fixed_100,proportional
n_ice,4.14,3.68,4.52,5.22,6.40,4.97
n_other,3.02,4.34,2.62,2.28,2.05,3.25
n_ovg,3.64,3.24,3.98,4.60,5.64,3.03


### 2.3 Select Final Allocation Strategy

In [9]:
# Select the allocation strategy to use for sampling
# Using the first option from FIXED_SAMPLES_OPTIONS by default
SELECTED_STRATEGY = f'fixed_{FIXED_SAMPLES_OPTIONS[0]}'

final_allocation = allocation_strategies[SELECTED_STRATEGY]

print(f"Selected Strategy: {SELECTED_STRATEGY}")
print("\nFinal Sample Allocation:")
for class_name, n_samples in zip(df_total.columns, final_allocation):
    print(f"  {class_name}: {n_samples} samples")
print(f"\nTotal: {sum(final_allocation)} samples")

Selected Strategy: fixed_300

Final Sample Allocation:
  n_ice: 300 samples
  n_other: 116 samples
  n_ovg: 300 samples

Total: 716 samples


---
## Step 3: Execute Stratified Random Sampling

Perform stratified random sampling from rasters to generate validation points.

**Note**: This section requires configuration of `SAMPLING_RASTERS` and setting `EXECUTE_SAMPLING = True`.

In [10]:
def perform_stratified_sampling(raster_paths, class_mapping, samples_per_class):
    """
    Perform stratified random sampling from rasters.
    
    Parameters
    ----------
    raster_paths : list
        List of raster file paths to sample from
    class_mapping : dict
        Dictionary mapping class values to class names
    samples_per_class : dict
        Dictionary mapping class names to number of samples
        
    Returns
    -------
    gpd.GeoDataFrame
        GeoDataFrame with sampled points (LON, LAT, class, raster, geometry)
    """
    print("Performing stratified random sampling...\n")
    
    # Load rasters
    rasters = []
    for raster_path in raster_paths:
        print(f"  Loading: {raster_path}")
        rasters.append(rioxr.open_rasterio(raster_path).squeeze())
    
    # Count pixels per class per raster for proportional allocation
    pixel_counts_per_raster = []
    for raster in rasters:
        counts = {}
        for class_val in class_mapping.keys():
            counts[class_val] = len(np.where(raster == class_val)[0])
        pixel_counts_per_raster.append(counts)
    
    # Distribute samples across rasters proportionally to pixel counts
    samples_distribution = {}  # {class_val: [n_raster0, n_raster1, ...]}
    
    for class_val, class_name in class_mapping.items():
        n_samples = samples_per_class[class_name]
        
        # Total pixels of this class across all rasters
        total_pixels = sum([counts[class_val] for counts in pixel_counts_per_raster])
        
        if total_pixels == 0:
            samples_distribution[class_val] = [0] * len(rasters)
            continue
        
        # Pixel counts per raster for this class
        pixels_per_raster = [counts[class_val] for counts in pixel_counts_per_raster]
        
        # Probabilities proportional to pixel counts
        probs = [n / total_pixels for n in pixels_per_raster]
        
        # Randomly allocate samples to rasters
        raster_choices = np.random.choice(
            range(len(rasters)),
            size=n_samples,
            p=probs
        )
        
        # Count samples per raster
        distribution = [np.sum(raster_choices == i) for i in range(len(rasters))]
        samples_distribution[class_val] = distribution
    
    # Sample points from each raster
    all_points = []
    
    for raster_idx, (raster, raster_path) in enumerate(zip(rasters, raster_paths)):
        print(f"\n  Sampling from raster {raster_idx + 1}/{len(rasters)}...")
        
        # Get raster filename without extension
        raster_filename = os.path.basename(raster_path)
        
        points = []
        classes = []
        raster_names = []
        
        for class_val in class_mapping.keys():
            n_samples = samples_distribution[class_val][raster_idx]
            
            if n_samples == 0:
                continue
            
            # Find all pixels of this class
            class_pixels = np.where(raster == class_val)
            
            if len(class_pixels[0]) == 0:
                print(f"    Warning: No pixels found for class {class_val}")
                continue
            
            # Sample random indices
            indices = sample(
                list(zip(class_pixels[0], class_pixels[1])),
                min(n_samples, len(class_pixels[0]))
            )
            
            # Get coordinates
            y_indices, x_indices = zip(*indices)
            x_coords = raster.x[np.asarray(x_indices)]
            y_coords = raster.y[np.asarray(y_indices)]
            
            # Create points
            points.extend([Point(x, y) for x, y in zip(x_coords, y_coords)])
            classes.extend([class_val] * len(indices))
            raster_names.extend([raster_filename] * len(indices))
            
            print(f"    Class {class_val}: {len(indices)} points")
        
        if points:
            # Create GeoDataFrame and convert to EPSG:4326
            gdf = gpd.GeoDataFrame(
                {'geometry': points, 'class': classes, 'raster': raster_names},
                crs=raster.rio.crs
            )
            all_points.append(gdf.to_crs(CRS.from_epsg(4326)))
    
    # Combine all points
    print("\n  Combining and shuffling points...")
    points_gdf = gpd.GeoDataFrame(pd.concat(all_points, ignore_index=True))
    points_gdf['LON'] = points_gdf.geometry.x
    points_gdf['LAT'] = points_gdf.geometry.y
    
    # Randomize order
    points_gdf = points_gdf.sample(frac=1).reset_index(drop=True)
    
    print(f"\n  Total points sampled: {len(points_gdf)}")
    
    return points_gdf

In [ ]:
# Convert final allocation to dictionary
samples_per_class_dict = dict(zip(df_total.columns, final_allocation))

# Perform sampling using the same rasters from Step 1
validation_points = perform_stratified_sampling(
    RASTER_FILES,
    CLASS_MAPPING,
    samples_per_class_dict
)

# Display results
print("\nValidation Points (first 10):")
display(validation_points.head(10))

print("\nClass Distribution:")
print(validation_points['class'].value_counts().sort_index())

print("\nRaster Distribution:")
print(validation_points['raster'].value_counts())

# Save if requested
if SAVE_POINTS:
    # Save as CSV (without geometry column for compatibility)
    csv_data = validation_points.drop(columns=['geometry'])
    csv_data.to_csv(OUTPUT_FILENAME, index_label='PLOTID')
    print(f"\nPoints saved to CSV: {OUTPUT_FILENAME}")
    
    # Save as shapefile
    shapefile_name = OUTPUT_FILENAME.replace('.csv', '.shp')
    validation_points.to_file(shapefile_name, driver='ESRI Shapefile')
    print(f"Points saved to Shapefile: {shapefile_name}")


---
## Summary

This notebook provides a complete workflow for stratified random sampling following best practices:

1. **Pixel counting** gives us the class distribution in the map
2. **Sample size calculation** determines how many points we need for desired accuracy
3. **Sample allocation** balances between proportional and equal strategies
4. **Stratified sampling** generates validation points ready for platforms like Collect Earth Online

The modular design allows easy customization of parameters and reuse of individual components.